# 1.基础容器

## 1.1 List
list 动态数组。它适合按下标随机访问、在尾部追加/删除，以及保持元素的先后顺序。在中间插入或删除元素时，后面的引用要整体挪动，代价是 O(n)。

### 1.1.1 常用操作

In [1]:
fruits = ["apple", "banana", "orange"]

print(fruits[0])       # apple
print(fruits[-1])      # orange，负数下标从末尾数
print(fruits[1:])      # ['banana', 'orange']
print(fruits[::-1])    # ['orange', 'banana', 'apple']，切片会创建新列表

apple
orange
['banana', 'orange']
['orange', 'banana', 'apple']


In [2]:
nums = [3, 1, 2]

nums.append(4)          # 尾部追加一个元素
nums.extend([5, 6])     # 把另一个可迭代对象的元素逐个追加
nums.insert(0, 0)       # 在下标 0 前插入
print(nums)             # [0, 3, 1, 2, 4, 5, 6]

nums.remove(2)          # 删除第一个值为 2 的元素；找不到会抛 ValueError
last = nums.pop()       # 删除并返回最后一个元素
first = nums.pop(0)     # 删除并返回下标 0 的元素，O(n)
del nums[1]             # 按下标删除，不返回被删元素
print(nums, last, first)

[0, 3, 1, 2, 4, 5, 6]
[3, 4, 5] 6 0


查/改列表时常用的小方法:

In [3]:
nums = [1, 2, 2, 3]

print(nums.count(2))    # 2，统计出现次数
print(nums.index(2))    # 1，第一个 2 的下标；不存在会抛 ValueError

nums.reverse()          # 原地反转，返回 None
print(nums)             # [3, 2, 2, 1]

2
1
[3, 2, 2, 1]


### 1.1.2 排序：sort 与 sorted

In [5]:
people = [
    {"name": "Ada", "age": 36},
    {"name": "Bob", "age": 20},
    {"name": "Cara", "age": 36},
]

# 先按年龄从大到小，年龄相同再按名字升序
people.sort(key=lambda p: (-p["age"], p["name"]))
print(people)
# [{'name': 'Ada', 'age': 36}, {'name': 'Cara', 'age': 36}, {'name': 'Bob', 'age': 20}]

new_people = sorted(people, key=lambda p: p["age"], reverse=True)# sorted 返回新列表

[{'name': 'Ada', 'age': 36}, {'name': 'Cara', 'age': 36}, {'name': 'Bob', 'age': 20}]


### 1.1.3 解包

In [6]:
first, *middle, last = [1, 2, 3, 4, 5]
print(first, middle, last)      # 1 [2, 3, 4] 5

1 [2, 3, 4] 5


### 1.1.4 浅拷贝与深拷贝
b = a 不会复制列表，只是让两个变量指向同一个对象。a[:]、a.copy()、list(a) 都是浅拷贝：只复制最外层，内层元素仍然共享。

In [7]:
original = [[1, 2], [3, 4]]
shallow = original.copy()

shallow[0].append(99)
print(original)   # [[1, 2, 99], [3, 4]]，内层列表仍然共享

import copy
deep = copy.deepcopy(original)
deep[0].append(100)
print(original)   # 不受影响

[[1, 2, 99], [3, 4]]
[[1, 2, 99], [3, 4]]


## 1.2 Dict：键值映射与哈希表

dict 是“键 → 值”的映射，底层是哈希表，平均查找、插入、删除都是 O(1)。

### 1.2.1 更新与删除

In [9]:
stu = {"name": "kk", "gender": "male", "age": 18}

# update 更新
stu.update({"name": "en", "grade": "A"})

# 删除
theme = stu.pop("gender")          # 删除并返回
font = stu.pop("missing", "nono") # 缺失时返回默认值
last_item = stu.popitem()         # 删除并返回最后一个键值对
del stu["age"]

### 1.2.2 遍历与字典推导式

In [10]:
scores = {"Ada": 90, "Bob": 85, "Cara": 95}

for name, score in scores.items():
    print(name, score)

# 只遍历键
for name in scores:
    print(name)

# 只遍历值
for score in scores.values():
    print(score)

# 字典推导式：分数加 5
updated = {name: score + 5 for name, score in scores.items()}
print(updated)         # {'Ada': 95, 'Bob': 90, 'Cara': 100}

Ada 90
Bob 85
Cara 95
Ada
Bob
Cara
90
85
95
{'Ada': 95, 'Bob': 90, 'Cara': 100}


# 2.函数：Lambda 匿名函数、Decorator 装饰器

## 2.1 Lambda：只包含一个表达式的匿名函数

它返回一个函数对象，函数体只能是一个表达式，不能写 if 语句、赋值、循环、return 等语句

In [11]:
add = lambda x, y: x + y
print(add(2, 3))       # 5

# 更常见的用法：作为排序的 key
people = [("Ada", 36), ("Bob", 20), ("Cara", 36)]
people.sort(key=lambda item: (-item[1], item[0]))
print(people)          # [('Ada', 36), ('Cara', 36), ('Bob', 20)]

# 按绝对值排序
nums = [-3, 1, -2]
print(sorted(nums, key=lambda x: abs(x)))   # [1, -2, -3]

5
[('Ada', 36), ('Cara', 36), ('Bob', 20)]
[1, -2, -3]


## 2.2 Decorator：在不改原函数代码的前提下增强函数

装饰器本质上是一个“接收函数、返回新函数”的高阶函数。@decorator 只是语法糖

### 2.2.1 基础使用

In [13]:
# 1.传统方式：
# 装饰器
def check(fn):
    def inner():
        print("人脸识别")   # 额外功能
        fn()
    return inner
# 被装饰的函数
def load():
    print("登陆")
# 使用装饰器装饰函数
load = check(load)
load()              # 人脸识别  登陆

# 2.语法糖
@check
def load():
    print("登陆")
load()              # 人脸识别  登陆



人脸识别
登陆
人脸识别
登陆


### 2.2.2 多个装饰器的顺序
装饰器从下往上包裹，运行时从上往下进入。

In [15]:
def a(func):
    def wrapper(*args, **kwargs):
        print("进入 a")
        result = func(*args, **kwargs)
        print("离开 a")
        return result
    return wrapper

def b(func):
    def wrapper(*args, **kwargs):
        print("进入 b")
        result = func(*args, **kwargs)
        print("离开 b")
        return result
    return wrapper

@a
@b
def f():
    print("执行 f")

f()
# 进入 a
# 进入 b
# 执行 f
# 离开 b
# 离开 a

进入 a
进入 b
执行 f
离开 b
离开 a


### 2.2.3 带有参数的装饰器

In [16]:
# 定义装饰器
def logging(flag):
    # 外部函数
    def decorator(fn):
        # 内部函数
        def inner(num1, num2):
            if flag == '+':
                print("在算加法")
            elif flag == '-':
                print("在算减法")
            result = fn(num1, num2)
            return result
        # 返回内部函数
        return inner
    # 返回装饰器
    return decorator
# 被带有参数的装饰器装饰的函数
@logging('+')
def add(num1, num2):
    return num1 + num2
@logging('-')
def sub(num1, num2):
    return num1 - num2

print(add(1, 2))
print(sub(3, 1))


在算加法
3
在算减法
2


# 3. 面向对象：Class、Magic Methods 与 OOP 思想

## 3.1 Class 类 实例方法、类方法、静态方法

In [17]:
class User:
    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age = age

    def introduce(self) -> str:              # 实例方法：需要 self
        return f"我是 {self.name}，{self.age} 岁"

    @classmethod
    def from_string(cls, raw: str) -> "User":  # 类方法：需要 cls
        name, age = raw.split(",")
        return cls(name.strip(), int(age))

    @staticmethod
    def is_valid_age(age: int) -> bool:      # 静态方法：不需要 self/cls
        return 0 <= age <= 150

## 3.2 封装与 @property

In [18]:
class Account:
    def __init__(self):
        self._balance = 0

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("存款金额必须大于 0")
        self._balance += amount


account = Account()
account.deposit(100)
print(account.balance)    # 100
# account.balance = 200   # 没有 setter，不能直接改

25
30


## 3.3 继承、super() 与组合

In [19]:
class Animal:
    def __init__(self, name: str) -> None:
        self.name = name

    def speak(self) -> str:
        raise NotImplementedError

class Cat(Animal):
    def speak(self) -> str:
        return f"{self.name}: 喵"

class Puppy(Animal):
    def __init__(self, name: str, trained: bool = False) -> None:
        super().__init__(name)        # 调用父类初始化
        self.trained = trained

    def speak(self) -> str:
        return f"{self.name}: 汪"

for animal in [Cat("咪咪"), Puppy("旺财")]:
    print(animal.speak())

咪咪: 喵
旺财: 汪


## 3.4 多态与抽象接口

In [20]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass

class Circle(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

def total_area(shapes: list[Shape]) -> float:
    return sum(shape.area() for shape in shapes)

print(total_area([Circle(1), Rectangle(2, 3)]))

9.14159


## 3.5 OOP 思想(网络搜索结果)
1、让对象维护自己的不变量：余额不能为负、温度不能低于绝对零度，这些规则应该放在类内部，而不是让每个调用者记得检查。

2、接口小而稳定：对外暴露越少，内部越容易改。不要把每个属性都做成 getter/setter。

3、优先组合，谨慎继承：继承是强耦合关系；组合更容易替换和测试。

4、依赖抽象，不依赖具体实现：函数参数写 Shape/Protocol，而不是写死 Circle。

5、一个类只负责一件事：如果类名里出现“并且”，或者方法分成明显无关的几组，就该拆。

## 3.6 魔法方法

|方法|什么时候自动调用|作用|
|----|----|----|
|__init__|Playlist("通勤")|初始化对象|
|__str__|print(obj)、str(obj)|给用户看的文字|
|__repr__|repr(obj)、调试器|给开发者看的文字|
|__len__|len(obj)|返回长度|
|__getitem__|obj[0]|支持下标|
|__iter__|for x in obj|支持遍历|
|__eq__|obj1 == obj2|自定义相等规则|
|__add__|obj1 + obj2|自定义加法|

# 4.文本处理：re 正则表达式

正则用来“按模式找字符串”。最常用的场景：校验格式、提取一段内容、替换、切分。

## 4.1 几个最常用的函数

|函数|用途|
|----|----|
|re.match(pattern, text)|从开头尝试匹配，不要求整串匹配|
|re.search(pattern, text)|找第一个匹配|
|re.fullmatch(pattern, text)|整个字符串必须完全匹配，适合校验|
|re.findall(pattern, text)|找出所有匹配，返回列表|
|re.sub(pattern, new, text)|替换|
|re.split(pattern, text)|按模式切分|

In [22]:
import re

text = "订单 A123 金额 45 元"

print(re.search(r"[A-Z]\d{3}", text).group())  # A123
print(re.findall(r"\d+", text))                # ['123', '45']
print(re.sub(r"\d+", "X", text))               # 订单 AX 金额 X 元
print(re.split(r"\s+", text))                  # ['订单', 'A123', '金额', '45', '元']

A123
['123', '45']
订单 AX 金额 X 元
['订单', 'A123', '金额', '45', '元']


## 4.2 常用符号

|符号|含义|
|----|----|
|.|任意一个字符（默认不含换行）|
|\d|数字|
|\w|字母、数字、下划线|
|\s|空白|
|*|出现 0 次或多次|
|+|出现 1 次或多次|
|?|出现 0 次或 1 次|
|?|出现 0 次或 1 次|
|{3}|正好 3 次|
|{2,5}|2 到 5 次|
|a|b|c|a、b、c 中任意一个|
|[^abc]|不是 a、b、c|
|^ / $|开头 / 结尾|
|(...)|分组|
|re.I|忽略大小写|

## 4.3 常用语法

|语法|含义|
|----|----|
|.|	除换行外的任意字符；re.S 时包括换行|
|\\d / \\D|数字 / 非数字|
|\\w / \\W|单词字符（字母、数字、下划线，Unicode 下更宽）/ 非单词字符|
|\\s / \\S|空白 / 非空白|
|a|b|c|a、b、c 中任意一个|
|[^abc]|不是 a、b、c|
|[a-z]|小写字母范围|
|*|前面的元素出现 0 次或多次|
|+|前面的元素出现 1 次或多次|
|?|前面的元素出现 0 次或 1 次|
|{m}|恰好 m 次|
|{m,n}|m 到 n 次|
|*?、+?、??|非贪婪（懒惰）版本，尽量少匹配|
|<|字符串开头|
|$|字符串结尾；re.M 时也匹配每行开头/结尾|
|\b|单词边界|
|\B|非单词边界|
|(...)|捕获组|
|?:...|非捕获组，只分组不保存|
|(?P<name>...)|命名捕获组|
|(?P=name)|反向引用命名组|
|(?=...)|正向前瞻|
|(?!...)|负向前瞻|
|(?i)|内联忽略大小写标志|

# 5. 代码格式：列表推导式与 Type Hint

## 5.1 列表推导式

In [23]:
squares = [x * x for x in range(5)]
print(squares)              # [0, 1, 4, 9, 16]

evens = [x for x in range(10) if x % 2 == 0]
print(evens)                # [0, 2, 4, 6, 8]

nums = [1, 2, 3, 4]
result = [x if x % 2 == 0 else -x for x in nums]
print(result)               # [-1, 2, -3, 4]

scores = {"Ada": 90, "Bob": 85}
passed = {name: score for name, score in scores.items() if score >= 90}
print(passed)               # {'Ada': 90}

letters = {word[0] for word in ["apple", "avocado", "banana"]}
print(letters)              # {'a', 'b'}

[0, 1, 4, 9, 16]
[0, 2, 4, 6, 8]
[-1, 2, -3, 4]
{'Ada': 90}
{'b', 'a'}


## 5.2 Type Hint 类型注释

类型注释说明“这个变量/参数/返回值应该是什么类型”。它主要帮助 IDE 和类型检查工具，运行时不会自动报错。

In [25]:
def average(nums: list[float]) -> float:
    return sum(nums) / len(nums)


def find_user(user_id: int) -> str | None:
    if user_id <= 0:
        return None
    return f"user-{user_id}"


count: int = 0
names: list[str] = []
scores: dict[str, int] = {}
point: tuple[int, int] = (1, 2)

常见写法：

|写法| 表示 |
|----|----|
|int、str、float、bool| 基本类型 |
|list[int]| 整数列表 |
|dict[str, int]| 键是字符串、值是整数 |
|tuple[int, str]| 两个元素分别是整数、字符串|
|set[str]| 字符串集合 |
str | None字符串或 None |

# 6.Generator 与 yield

普通函数 return 一次就结束；生成器函数用 yield，可以“产出一个值，暂停，等下一次再继续”。

In [26]:
# 最简单的生成器
def countdown(n):
    while n > 0:
        yield n
        n -= 1


g = countdown(3)
print(next(g))    # 3
print(next(g))    # 2
print(next(g))    # 1

3
2
1


In [27]:
# 用生成器读大文件
def read_numbers(path):
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                yield int(line)

In [28]:
# 生成器表达式
squares = (x * x for x in range(5))
print(next(squares))     # 0
print(list(squares))     # [1, 4, 9, 16]

# 直接求和，不创建完整列表
print(sum(x * x for x in range(5)))   # 30

0
[1, 4, 9, 16]
30


In [29]:
# yield from 可以把另一个可迭代对象的值逐个交出去，递归展平时很好用。
def flatten(items):
    for item in items:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item


print(list(flatten([1, [2, [3, 4]], 5])))   # [1, 2, 3, 4, 5]

[1, 2, 3, 4, 5]
